# Mathbreakers: Glitch Rally — raw SLM candidate generation

This free-Colab notebook runs the final v7.1 distractor adapter across the validated original game-question bank. It produces **raw review candidates**, never approved game content.

Before opening this notebook in Colab, build the exact upload bundle from the current local working tree:

```bash
.venv/bin/python -m src.game_colab_bundle --output glitch_rally_colab_bundle.zip
```

In Colab, choose **Runtime → Change runtime type → T4 GPU**, run all cells, and upload that zip when prompted. The notebook verifies the allowlist, manifest, every source hash, and the frozen 140-row holdout before importing any uploaded code.

In [ ]:
%pip install -q "unsloth==2026.7.1"

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib
import io
import json
import os
import stat
import sys
import tempfile
import zipfile

BUNDLE_SCHEMA_VERSION = "glitch-rally-colab-bundle-v1"
BUNDLE_MANIFEST_PATH = "glitch_rally_bundle_manifest.json"
BUNDLE_FILES = (
    "data/game/questions_v1.jsonl",
    "data/processed/eval_heldout.jsonl",
    "src/__init__.py",
    "src/buggy_procedures.py",
    "src/config.py",
    "src/consistency.py",
    "src/game_colab_backend.py",
    "src/game_candidate_generation.py",
    "src/game_content.py",
    "src/prompts.py",
    "src/text_utils.py",
)
FROZEN_HOLDOUT_COUNT = 140
FROZEN_HOLDOUT_SHA256 = "47ce1e1b85ebaae0782f0aed32fa12bb6ec0fd4498ed71c75cf3e4aff5135693"
MAX_MEMBER_BYTES = 10 * 1024 * 1024
MAX_BUNDLE_BYTES = 25 * 1024 * 1024

def stable_json_sha256(value):
    payload = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise RuntimeError(f"Duplicate JSON key in bundle: {key}")
        result[key] = value
    return result

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one glitch_rally_colab_bundle.zip file.")
upload_name, bundle_bytes = next(iter(uploaded.items()))
if not upload_name.lower().endswith(".zip"):
    raise RuntimeError("The uploaded Colab source bundle must be a zip file.")

with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    infos = archive.infolist()
    names = [info.filename for info in infos]
    for name in names:
        path = PurePosixPath(name)
        if (
            not name
            or "\\" in name
            or path.is_absolute()
            or any(part in {"", ".", ".."} for part in path.parts)
        ):
            raise RuntimeError(f"Unsafe bundle path: {name!r}")
    if len(names) != len(set(names)):
        raise RuntimeError("The bundle contains duplicate member names.")
    expected_names = set(BUNDLE_FILES) | {BUNDLE_MANIFEST_PATH}
    if set(names) != expected_names:
        raise RuntimeError("The bundle member allowlist does not match.")
    if any(info.is_dir() for info in infos):
        raise RuntimeError("The bundle may contain files only.")
    if any(stat.S_ISLNK(info.external_attr >> 16) for info in infos):
        raise RuntimeError("The bundle may not contain symbolic links.")
    if any(info.flag_bits & 0x1 for info in infos):
        raise RuntimeError("The bundle may not contain encrypted files.")
    if any(info.file_size > MAX_MEMBER_BYTES for info in infos):
        raise RuntimeError("A bundle member exceeds the size limit.")
    if sum(info.file_size for info in infos) > MAX_BUNDLE_BYTES:
        raise RuntimeError("The bundle exceeds the total size limit.")

    manifest = json.loads(
        archive.read(BUNDLE_MANIFEST_PATH),
        object_pairs_hook=reject_duplicate_keys,
    )
    required_manifest_fields = {
        "schema_version", "bundle_id", "files",
        "frozen_holdout_count", "frozen_holdout_sha256",
        "generator_source_sha256", "backend_source_sha256",
    }
    if not isinstance(manifest, dict) or set(manifest) != required_manifest_fields:
        raise RuntimeError("The bundle manifest schema is invalid.")
    if manifest["schema_version"] != BUNDLE_SCHEMA_VERSION:
        raise RuntimeError("The bundle manifest version is not supported.")

    listed_paths = []
    for item in manifest["files"]:
        if not isinstance(item, dict) or set(item) != {"path", "size", "sha256"}:
            raise RuntimeError("A bundle manifest file entry is invalid.")
        relative = item["path"]
        if relative not in BUNDLE_FILES or relative in listed_paths:
            raise RuntimeError(f"Unexpected or duplicate manifest path: {relative!r}")
        listed_paths.append(relative)
        payload = archive.read(relative)
        if item["size"] != len(payload):
            raise RuntimeError(f"Bundle size mismatch: {relative}")
        if item["sha256"] != hashlib.sha256(payload).hexdigest():
            raise RuntimeError(f"Bundle hash mismatch: {relative}")
    if tuple(listed_paths) != BUNDLE_FILES:
        raise RuntimeError("The manifest file order or coverage is invalid.")

    manifest_core = {key: value for key, value in manifest.items() if key != "bundle_id"}
    expected_bundle_id = f"bundle:v1:{stable_json_sha256(manifest_core)}"
    if manifest["bundle_id"] != expected_bundle_id:
        raise RuntimeError("The bundle identity hash does not match.")

    holdout_text = archive.read("data/processed/eval_heldout.jsonl").decode("utf-8")
    holdout = [
        json.loads(line, object_pairs_hook=reject_duplicate_keys)
        for line in holdout_text.splitlines() if line.strip()
    ]
    if (
        manifest["frozen_holdout_count"] != FROZEN_HOLDOUT_COUNT
        or len(holdout) != FROZEN_HOLDOUT_COUNT
        or manifest["frozen_holdout_sha256"] != FROZEN_HOLDOUT_SHA256
        or stable_json_sha256(holdout) != FROZEN_HOLDOUT_SHA256
    ):
        raise RuntimeError("The frozen holdout receipt does not match.")
    generator_bytes = archive.read("src/game_candidate_generation.py")
    if manifest["generator_source_sha256"] != hashlib.sha256(generator_bytes).hexdigest():
        raise RuntimeError("The generator source hash does not match.")
    backend_bytes = archive.read("src/game_colab_backend.py")
    if manifest["backend_source_sha256"] != hashlib.sha256(backend_bytes).hexdigest():
        raise RuntimeError("The Colab backend source hash does not match.")

    REPO_DIR = Path(tempfile.mkdtemp(prefix="glitch-rally-verified-"))
    for relative in BUNDLE_FILES:
        target = REPO_DIR.joinpath(*PurePosixPath(relative).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(relative))

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
BUNDLE_MANIFEST = manifest
print(f"Verified {manifest['bundle_id']} in fresh directory {REPO_DIR}")

## Run identity

Keep `RUN_ID` unchanged when resuming the same output in this Colab runtime. Use a non-identifying lowercase label matching `[a-z0-9][a-z0-9._-]{2,79}`—never a learner name or email. Change it and the output filename before deliberately starting another run. Mutable Hugging Face labels such as `main` are resolved to immutable 40-character commits before model loading.

In [ ]:
RUN_ID = "glitch-rally-v1-owner-review-01"
MODEL_ID = "unsloth/Qwen3-4B-bnb-4bit"
MODEL_REQUESTED_REVISION = "main"
ADAPTER_ID = "j2ampn/qwen3-4b-distractor-lora-v7"
ADAPTER_REQUESTED_REVISION = "main"
QUESTIONS_PATH = REPO_DIR / "data/game/questions_v1.jsonl"
HOLDOUT_PATH = REPO_DIR / "data/processed/eval_heldout.jsonl"
OUTPUT_PATH = Path("/content/glitch_rally_raw_candidates_v1.jsonl")
RUN_MANIFEST_PATH = Path("/content/glitch_rally_generation_run_v1.json")
RESUME = True

In [ ]:
from src.game_colab_backend import (
    BACKEND_SOURCE_SHA256,
    load_pinned_colab_backend,
)
from src.game_candidate_generation import (
    GENERATION_PARAMETERS,
    GENERATOR_SOURCE_SHA256,
    GenerationProvenance,
    generate_candidate_batch,
    load_validated_question_batch,
)

if GENERATOR_SOURCE_SHA256 != BUNDLE_MANIFEST["generator_source_sha256"]:
    raise RuntimeError("Imported generator does not match the verified bundle.")
if BACKEND_SOURCE_SHA256 != BUNDLE_MANIFEST["backend_source_sha256"]:
    raise RuntimeError("Imported Colab backend does not match the verified bundle.")

questions = load_validated_question_batch(QUESTIONS_PATH, HOLDOUT_PATH)
if (
    questions.holdout_count != BUNDLE_MANIFEST["frozen_holdout_count"]
    or questions.holdout_sha256 != BUNDLE_MANIFEST["frozen_holdout_sha256"]
):
    raise RuntimeError("Question receipt does not match the verified bundle.")
print(f"Validated {len(questions)} original, non-holdout questions.")

In [ ]:
loaded_backend = load_pinned_colab_backend(
    model_id=MODEL_ID,
    model_requested_revision=MODEL_REQUESTED_REVISION,
    adapter_id=ADAPTER_ID,
    adapter_requested_revision=ADAPTER_REQUESTED_REVISION,
)
resolved_model_revision = loaded_backend.model_revision
resolved_adapter_revision = loaded_backend.adapter_revision
print(f"Base revision:    {resolved_model_revision}")
print(f"Adapter revision: {resolved_adapter_revision}")
print("Pinned base and v7.1 adapter loaded once by the verified backend.")

In [ ]:
print(f"Verified backend source: {BACKEND_SOURCE_SHA256}")

In [ ]:
provenance = GenerationProvenance(
    run_id=RUN_ID,
    model_id=MODEL_ID,
    model_revision=resolved_model_revision,
    adapter_id=ADAPTER_ID,
    adapter_revision=resolved_adapter_revision,
    backend_source_sha256=BACKEND_SOURCE_SHA256,
)
records = generate_candidate_batch(
    questions=questions,
    output_path=OUTPUT_PATH,
    backend=loaded_backend,
    provenance=provenance,
    resume=RESUME,
)
print(f"Saved {len(records)} raw candidates to {OUTPUT_PATH}")

In [ ]:
from importlib.metadata import PackageNotFoundError, version

runtime_versions = {}
for package in ("unsloth", "torch", "transformers", "peft", "huggingface_hub"):
    try:
        runtime_versions[package] = version(package)
    except PackageNotFoundError:
        runtime_versions[package] = "not-installed"
run_manifest = {
    "schema_version": "glitch-rally-generation-run-v1",
    "run_id": RUN_ID,
    "bundle_id": BUNDLE_MANIFEST["bundle_id"],
    "generator_source_sha256": GENERATOR_SOURCE_SHA256,
    "backend_source_sha256": BACKEND_SOURCE_SHA256,
    "model_id": MODEL_ID,
    "model_revision": resolved_model_revision,
    "adapter_id": ADAPTER_ID,
    "adapter_revision": resolved_adapter_revision,
    "generation_parameters": dict(GENERATION_PARAMETERS),
    "source_batch_sha256": questions.source_batch_sha256,
    "candidate_count": len(records),
    "output_sha256": hashlib.sha256(OUTPUT_PATH.read_bytes()).hexdigest(),
    "runtime_versions": runtime_versions,
}
RUN_MANIFEST_PATH.write_text(
    json.dumps(run_manifest, ensure_ascii=False, sort_keys=True, indent=2) + "\n",
    encoding="utf-8",
)
files.download(str(OUTPUT_PATH))
files.download(str(RUN_MANIFEST_PATH))